In [1]:
import os
import re
from pathlib import Path
from typing import List
import aiofiles
import aiohttp
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console
from autogen_core.memory import Memory, MemoryContent, MemoryMimeType
from autogen_ext.memory.mem0 import Mem0Memory
from autogen_ext.memory.chromadb import (
    ChromaDBVectorMemory,
    PersistentChromaDBVectorMemoryConfig,
)
# from mem0 import MemoryClient
from autogen_ext.models.openai import OpenAIChatCompletionClient
import os
from dotenv import load_dotenv
import time

In [2]:
load_dotenv()

True

In [3]:
api_key = os.environ.get("AICREDITS_API_KEY")
base_url = os.environ.get("BASE_URL")
mem0_api_key = os.getenv("MEM0_API_KEY")

if not api_key:
    raise ValueError("Please set the OpenAI_API_KEY")

if not base_url:
    raise ValueError("Please set the BASE_URL")

if not mem0_api_key:
    raise ValueError("Please set the BASE_URL")

### Mem0 Memory Example

In [4]:
async def run_mem0_preference_example():
    model_client = OpenAIChatCompletionClient(
        model="gpt-4o",
        api_key=api_key,
        base_url= base_url,
    )

    mem0_memory = Mem0Memory(
        user_id="prash12",
        is_cloud=True,
        api_key=mem0_api_key,
    )

    try:
        print("Attempting to add memories to cloud")

        messages = [
            {"role": "user", "content": "I am a vegetarian food and I love cooking."},
            {"role": "assistant", "content": "I will remember that preference is Vegetarian."},
        ]
        mem0_memory._client.add(
            messages,
            user_id="prash12",
            metadata={"category": "preferences", "type": "diet"},
            output_format="v1.1",
        )

        messages = [
            {"role": "user", "content": "I prefer weather information in metric units."},
            {"role": "assistant", "content": "I will remember that you prefer metric units in weather."},
        ]
        mem0_memory._client.add(
            messages,
            user_id="prash12",
            metadata={"category": "preferences", "type": "units"},
            output_format="v1.1",
        )

        messages = [
            {"role": "user", "content": "My favourite color is Black."},
            {"role": "assistant", "content": "I will remember that your favourite color is black."},
        ]
        mem0_memory._client.add(
            messages,
            user_id="prash12",
            metadata={"category": "preferences", "type": "color"},
            output_format="v1.1",
        )

        print("Memories added successfully")
    except Exception as e:
        print(f"Error adding memories: {e}")

    time.sleep(30)

    agent = AssistantAgent(
        name="assistant",
        model_client=model_client,
        memory=[mem0_memory],
        system_message=(
            "You are a helpful assistant that remembers user preferences "
            "and uses them to provide personalized responses."
        ),
    )

    try:
        result = await agent.run(task="What type of food do I like?")
        print("Agent Response:", result.messages[-1].content)
    except Exception as e:
        print(f"Error running agent: {e}")

In [5]:
await run_mem0_preference_example()

Attempting to add memories to cloud
Memories added successfully
Agent Response: Based on what I remember, you prefer vegetarian food and enjoy cooking. If you have any updates or if there’s something specific you’re craving, feel free to let me know!


In [6]:
async def get_weather(city: str, units: str = "imperial") -> str:
    if units == "imperial":
        return f"The weather in {city} is 73 °F and Sunny."
    elif units == "metric":
        return f"The weather in {city} is 23 °C and Sunny."
    else:
        return f"Sorry, I don't know the weather in {city}."

### Mem0 Memory with the Weather Assistant

In [7]:
mem0_memory = Mem0Memory(
    is_cloud=True,
    limit=5,
    api_key=mem0_api_key,
    user_id="prashant123",
)

await mem0_memory.add(
    MemoryContent(
        content="The weather should be in metric units",
        mime_type=MemoryMimeType.TEXT,
        metadata={"category": "preferences", "type": "units"},
    )
)

await mem0_memory.add(
    MemoryContent(
        content="Meal recipe must be vegan",
        mime_type=MemoryMimeType.TEXT,
        metadata={"category": "preferences", "type": "dietary"},
    )
)

assistant_agent = AssistantAgent(
    name="assistant_agent",
    model_client=OpenAIChatCompletionClient(
        model="gpt-4o",
        api_key=api_key,
        base_url= base_url,
    ),
    tools=[get_weather],
    memory=[mem0_memory],
)

print("Question: What are my dietary preferences?")
# print("Stored preferences include metric weather units and a vegan meal preference.")
stream = assistant_agent.run_stream(task="What are my dietary preferences?")
await Console(stream)

Question: What are my dietary preferences?
---------- TextMessage (user) ----------
What are my dietary preferences?
---------- MemoryQueryEvent (assistant_agent) ----------
[MemoryContent(content='User requires meal recipes to be vegan', mime_type='text/plain', metadata={'type': 'dietary', 'category': 'preferences', 'score': 0.1675, 'created_at': datetime.datetime(2026, 10, 25, 16, 38, 4, tzinfo=datetime.timezone.utc), 'updated_at': datetime.datetime(2026, 10, 25, 16, 38, 10, 894399, tzinfo=datetime.timezone.utc), 'categories': ['food', 'health', 'user_preferences']})]
---------- TextMessage (assistant_agent) ----------
Your dietary preference is vegan. You prefer meal recipes that do not include any animal products. Is there anything specific you'd like help with regarding your dietary preferences?


TaskResult(messages=[TextMessage(id='4442a16e-9f8a-481e-b680-d757a2893fb9', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 11, 13, 16, 24, 11, 806761, tzinfo=datetime.timezone.utc), content='What are my dietary preferences?', type='TextMessage'), MemoryQueryEvent(id='f8532bca-2fd4-4e44-8d99-c57c77006657', source='assistant_agent', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 11, 13, 16, 24, 13, 802973, tzinfo=datetime.timezone.utc), content=[MemoryContent(content='User requires meal recipes to be vegan', mime_type='text/plain', metadata={'type': 'dietary', 'category': 'preferences', 'score': 0.1675, 'created_at': datetime.datetime(2026, 10, 25, 16, 38, 4, tzinfo=datetime.timezone.utc), 'updated_at': datetime.datetime(2026, 10, 25, 16, 38, 10, 894399, tzinfo=datetime.timezone.utc), 'categories': ['food', 'health', 'user_preferences']})], type='MemoryQueryEvent'), TextMessage(id='a201acd5-d771-432e-8906-a278ba4842ea', source='assis

### Mem0 Retrieval Results

In [8]:
retrieved = mem0_memory._client.search(
    query="dietary preferences vegan meal",
    user_id="prashant123",
)
print(retrieved)

[{'id': 'f3257836-cb89-4dfd-aab7-4d888d1d49bc', 'memory': 'User requires meal recipes to be vegan', 'user_id': 'prashant123', 'score': 0.2976, 'categories': ['food', 'health', 'user_preferences'], 'metadata': {'type': 'dietary', 'category': 'preferences'}, 'updated_at': '2026-10-25T16:38:10.894399+00:00', 'expiration_date': None, 'created_at': '2026-10-25T16:38:04+00:00', 'structured_attributes': {'day': 25, 'hour': 16, 'year': 2026, 'month': 10, 'minute': 38, 'quarter': 4, 'is_weekend': True, 'day_of_week': 'sunday', 'day_of_year': 298, 'week_of_year': 43}}]


In [9]:
retrieved = mem0_memory._client.search(
    query="What units should I use for weather?",
    user_id="prashant123",
)

print(retrieved)

[{'id': '88a8dd31-0a59-4df3-a2f0-807af8823696', 'memory': 'User prefers weather information in metric units', 'user_id': 'prashant123', 'score': 0.2911, 'categories': ['user_preferences'], 'metadata': {'type': 'units', 'category': 'preferences'}, 'updated_at': '2026-10-25T16:38:07.888672+00:00', 'expiration_date': None, 'created_at': '2026-10-25T16:38:03+00:00', 'structured_attributes': {'day': 25, 'hour': 16, 'year': 2026, 'month': 10, 'minute': 38, 'quarter': 4, 'is_weekend': True, 'day_of_week': 'sunday', 'day_of_year': 298, 'week_of_year': 43}}]


## Simple RAG
The indexer fetches text from URLs or local files, 
- strips HTML tags when present, 
- splits text into fixed-size chunks, 
- and adds each chunk to the AutoGen memory.

In [10]:
class SimpleDocumentIndexer:
    """Basic document indexer for AutoGen Memory."""

    def __init__(self, memory: Memory, chunk_size: int = 1500) -> None:
        self.memory = memory
        self.chunk_size = chunk_size

    async def _fetch_content(self, source: str) -> str:
        """Fetch content from URL or file."""
        if source.startswith(("http://", "https://")):
            async with aiohttp.ClientSession() as session:
                async with session.get(source) as response:
                    return await response.text()
        async with aiofiles.open(source, "r", encoding="utf-8") as f:
            return await f.read()

    def _strip_html(self, text: str) -> str:
        """Remove HTML tags and normalize whitespace."""
        text = re.sub(r"<[^>]*>", " ", text)
        text = re.sub(r"\s+", " ", text)
        return text.strip()

    def _split_text(self, text: str) -> List[str]:
        """Split text into fixed-size chunks."""
        chunks: List[str] = []
        for i in range(0, len(text), self.chunk_size):
            chunks.append(text[i : i + self.chunk_size].strip())
        return chunks

    async def index_documents(self, sources: List[str]) -> int:
        """Index documents into memory."""
        total_chunks = 0
        for source in sources:
            try:
                content = await self._fetch_content(source)
                if "<" in content and ">" in content:
                    content = self._strip_html(content)
                chunks = self._split_text(content)
                for i, chunk in enumerate(chunks):
                    await self.memory.add(
                        MemoryContent(
                            content=chunk,
                            mime_type=MemoryMimeType.TEXT,
                            metadata={"source": source, "chunk_index": i},
                        )
                    )
                total_chunks += len(chunks)
            except Exception as e:
                print(f"Error indexing {source}: {str(e)}")
        return total_chunks

## ChromaDB Vector Memory
persistent ChromaDB vector memory, clears the existing collection, and indexes the documentation sources.

In [14]:
rag_memory = ChromaDBVectorMemory(
    config=PersistentChromaDBVectorMemoryConfig(
        collection_name="autogen_docs",
        persistence_path=os.path.join(str(Path.home()), ".chromadb_autogen"),
        k=3,
        score_threshold=0.3,
    )
)

await rag_memory.clear()

async def index_autogen_docs() -> None:
    indexer = SimpleDocumentIndexer(memory=rag_memory)
    sources = [
        "https://raw.githubusercontent.com/microsoft/autogen/main/README.md",
        "https://microsoft.github.io/autogen/dev/user-guide/agentchat-user-guide/tutorial/agents.html",
        "https://microsoft.github.io/autogen/dev/user-guide/agentchat-user-guide/tutorial/teams.html",
        "https://microsoft.github.io/autogen/dev/user-guide/agentchat-user-guide/tutorial/termination.html",
    ]
    chunks = await indexer.index_documents(sources)
    print(f"Indexed {chunks} chunks from {len(sources)} AutoGen documents")

await index_autogen_docs()

Indexed 73 chunks from 4 AutoGen documents


In [15]:
query = "How do I create an AssistantAgent in AutoGen?"

results = await rag_memory.query(query)

for i, result in enumerate(results.results, 1):
    print(f"\n--- Result {i} ---")
    print("Content:", result.content)
    print("Metadata:", result.metadata)


--- Result 1 ---
Content: now archived and no longer maintained by Microsoft; the architecture described below remains available for historical/reference purposes. **For new projects, we recommend [Microsoft Agent Framework](https://github.com/microsoft/agent-framework)**, which builds on the lessons learned from AutoGen with enterprise-grade support. The autogen _framework_ uses a layered and extensible design. Layers have clearly divided responsibilities and build on top of layers below. This design enables you to use the framework at different levels of abstraction, from high-level APIs to low-level components. - [Core API](./python/packages/autogen-core/) implements message passing, event-driven agents, and local and distributed runtime for flexibility and power. It also support cross-language support for .NET and Python. - [AgentChat API](./python/packages/autogen-agentchat/) implements a simpler but opinionated API for rapid prototyping. This API is built on top of the Core API 

## Combined Memory: Mem0 + ChromaDB

In [ ]:
combined_agent_rag_first = AssistantAgent(
    name="combined_agent_rag_first",
    model_client=OpenAIChatCompletionClient(
        model="gpt-4o",
        api_key=api_key,
        base_url=base_url,
    ),
    memory=[rag_memory, mem0_memory],
    system_message=(
        "You are a helpful assistant. Use the user's personal preferences "
        "and the retrieved AutoGen documentation to answer accurately."
    ),
)

combined_query_4 = "How do I create an AssistantAgent in AutoGen?"

result = await combined_agent_rag_first.run(task=combined_query_4)

for msg in result.messages:
    print(f"--- {type(msg).__name__} (source={msg.source}) ---")
    if type(msg).__name__ == "MemoryQueryEvent":
        for mc in msg.content:
            backend = "ChromaDB (doc chunk)" if "source" in mc.metadata else "Mem0 (personal fact)"
            print(f"[{backend}] score={mc.metadata.get('score')}")
            if "source" in mc.metadata:
                print("   source:", mc.metadata["source"], "| chunk_index:", mc.metadata["chunk_index"])
            print("   content:", str(mc.content)[:180].replace("\n", " "))
            print()
    else:
        print(msg.content)
    print()

--- TextMessage (source=user) ---
How do I create an AssistantAgent in AutoGen?

--- MemoryQueryEvent (source=combined_agent_rag_first) ---
[ChromaDB (doc chunk)] score=0.5546212494373322
   source: https://raw.githubusercontent.com/microsoft/autogen/main/README.md | chunk_index: 5
   content: now archived and no longer maintained by Microsoft; the architecture described below remains available for historical/reference purposes. **For new projects, we recommend [Microsof

[ChromaDB (doc chunk)] score=0.55439093708992
   source: https://raw.githubusercontent.com/microsoft/autogen/main/README.md | chunk_index: 2
   content: tion-guide.html) for detailed instructions on how to update your code and configurations. ```bash # Install AutoGen Studio for no-code GUI pip install -U "autogenstudio" ``` ## Qui

[ChromaDB (doc chunk)] score=0.5410679578781128
   source: https://raw.githubusercontent.com/microsoft/autogen/main/README.md | chunk_index: 1
   content: handling sensitive data. AutoGen 